# Assignment 3: Fine-tuning language models

In this assignment, you will perform supervised fine-tuning (SFT) of a small open LLM on an instruction tuning dataset. You will convert this dataset into instruction-response pairs, fine-tune a causal language model using LoRA (Low-Rank Adaptation), and evaluate it through prompted inference and comparison with other methods.

## Preliminaries

First, let's install the required libraries. If you are running in your own environment, make sure the following are installed:

- [Torch](https://docs.pytorch.org/docs/stable/index.html)
- [Transformers](https://huggingface.co/docs/transformers/index)
- [Datasets](https://huggingface.co/docs/datasets/index)
- [Evaluate](https://huggingface.co/docs/evaluate/en/index)
- [NLTK](https://www.nltk.org/api/nltk.html)
- [rouge_score](https://pypi.org/project/rouge-score/)

In a Colab notebook, most of them are already installed, except Evaluate and rouge_score.

In [2]:
%pip install evaluate rouge_score


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


We also set some configuration parameters.

Most importantly, you should select a language model to work with in this assignment and enter its HuggingFace identifier in the parameter `MODEL_NAME` below. In principle you can use any model that you want, but we recommend that you select a model that has not already been trained to follow instructions, so it should be a "pure" language model trained on raw text (similar to Assignments 1 and 2).

The selected model should be small enough to fit in your computational environment. We have verified that the 135-million parameter [`SmolLM2` model](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), developed by HuggingFace, can be used to solve this assignment in a Colab notebook (free tier, T4 GPU). If you run on a cluster, you can select a larger model (and probably see more interesting results).

We also define training and test set sizes here. Again, the values below have been set so that the assignment can be solved in Colab, and you can increase these sizes to improve the quality of the fine-tuned models.

In [3]:
import torch

SEED = 101
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_TRAIN_SAMPLES = 5000
MAX_TEST_SAMPLES = 400

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"

# Part 1: Preprocessing

### ⚙&nbsp; Task 1.1: Loading and inspecting the dataset

The dataset [SmolTalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) is a collection of instruction-response pairs designed for SFT of large language models for instruction following. This dataset consists of examples of user inputs with system responses.

You can load using the datasets from the HuggingFace repository as follows.

In [31]:
from datasets import load_dataset
from datasets import DatasetDict

smoltalk = load_dataset("HuggingFaceTB/smoltalk", 'all')

print(smoltalk)
print("One training example:")
print(smoltalk["train"][0])

DatasetDict({
    train: Dataset({
        features: ['messages', 'source'],
        num_rows: 1043917
    })
    test: Dataset({
        features: ['messages', 'source'],
        num_rows: 54948
    })
})
One training example:
{'messages': [{'content': 'The function \\( g(x) \\) satisfies the functional equation\n\\[ g(x + y) = g(x) + g(y) \\]\nfor all real numbers \\( x \\) and \\( y \\), and it is given that \\( g(3) = 4 \\). Find \\( g(10) \\).', 'role': 'user'}, {'content': 'Given the functional equation and the specific value \\( g(3) = 4 \\), we can find \\( g(1) \\) by using the equation multiple times:\n\\[\ng(3) = g(2) + g(1)\n\\]\n\\[\ng(2) = g(1) + g(1) = 2g(1)\n\\]\nThus,\n\\[\n4 = 2g(1) + g(1) = 3g(1)\n\\]\n\\[\ng(1) = \\frac{4}{3}\n\\]\nNow we can find \\( g(10) \\) using \\( g(1) \\):\n\\[\ng(10) = 10g(1) = 10 \\times \\frac{4}{3} = \\frac{40}{3}\n\\]\nHence, the value of \\( g(10) \\) is \\(\\boxed{\\frac{40}{3}}\\).', 'role': 'assistant'}], 'source': 'numina-cot-100k'

In order to make this assignment possible to solve in a restricted environment, we simplify the dataset a bit:
- We remove multi-turn chat dialogues from the dataset;
- We remove instances where the query or the answer is greater than a set maximum length;
- We keep a subset of the data for training and testing (by default 5000 and 400, respectively).

In [32]:
smoltalk_simplified = smoltalk.filter(lambda row: len(row['messages']) <= 3 and all(len(m['content']) <= 256 for m in row['messages']))
smoltalk_simplified = DatasetDict({
    "train": smoltalk_simplified["train"].select(range(MAX_TRAIN_SAMPLES)),
    "test": smoltalk_simplified["test"].select(range(MAX_TEST_SAMPLES)),
})

In [33]:
smoltalk_simplified

DatasetDict({
    train: Dataset({
        features: ['messages', 'source'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['messages', 'source'],
        num_rows: 400
    })
})

Print some examples from the dataset so that you understand the format.

Key points you need to note here: each example from the training or test set consists of a sequence of messages. The number of messages in each example will be 2 or 3, because we removed multi-turn chat dialogues in the previous step. Each message is associated with a `role` label:
- `user`: an example of something the user might write.
- `assistant`: an example of an output an LLM could be expected to produce, given the input.
- `system`: a *system prompt* that gives guidelines for the general behavior of the LLM's behavior.

All examples in the dataset include a user input and an assistant output, but the system prompt is not available in all of the examples.

In [34]:
smoltalk_simplified['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting'}

### 🎓&nbsp; Task 1.2: Formatting the data for instruction tuning

Define a function `format_input_output` that converts an example from the dataset into an input/output pair that we can use to fine-tune the LLM.

You are free to design the format. The following document gives some examples that have been used by different instruction-following LLMs including Llama and Mistral: https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats

The later stages of our preprocessing pipeline expect that this function returns an object containing two parts: the `prompt` (what goes into the LLM before generating anything) and the `response` (what the LLM is expected to generate).

## My relection:
it is important to decide on some specific symbols for instructions and you can decide, somebody decided on html.


In [35]:
def format_input_output(example):
    # `messages` is a list of messages, each with a `content` string and a `role`.
    messages = example['messages']

    system_text = ""
    user_text = ""
    assistant_text = ""

    for m in messages:
        if m["role"] == "system":
            system_text = m["content"]
        elif m["role"] == "user":
            user_text = m["content"]
        elif m["role"] == "assistant":
            assistant_text = m["content"]

    prompt = "### Instruction\n"
    if system_text != "":
        prompt = prompt + system_text + "\n"
    prompt = prompt + user_text + "\n"
    prompt = prompt + "### Response\n"

    response = assistant_text

    return {"prompt": prompt, "response": response}

Apply the function you implemented to the dataset as a whole.

In [36]:
ds_sft = smoltalk_simplified.map(format_input_output)

Then verify that the dataset now contains the new fields you created.

In [37]:
print("BEFORE format_input_output:")
print(smoltalk_simplified['train'][0])

print("\nAFTER format_input_output:")
print(ds_sft['train'][0])

BEFORE format_input_output:
{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.", 'role': 'system'}, {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.', 'role': 'user'}, {'content': 'The chef made more food after the restaurant ran out.', 'role': 'assistant'}], 'source': 'explore-instruct-rewriting'}

AFTER format_input_output:
{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.", 'role': 'system'}, {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.', 'role': 'user'}, {'content': 'The chef made more food after the restaurant ran out.', 'role': 'assistant'}], 'source': 'explore-instruct-rewriting', 'prompt': "### Instruc

### ⚙&nbsp; Task 1.3: Tokenizing the dataset

We will now prepare the format required by the HuggingFace Trainer.

We first load the tokenizer for our selected model:

In [38]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [39]:
print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)
print("vocab size:", tokenizer.vocab_size)
print(tokenizer("Hello world!"))

print("Total tokens in vocab:", tokenizer.vocab_size)

print("\nFirst 5 tokens:")
for tid in range(5):
    print(tid, "->", repr(tokenizer.convert_ids_to_tokens(tid)))

print("\nLast 5 tokens:")
for tid in range(tokenizer.vocab_size - 5, tokenizer.vocab_size):
    print(tid, "->", repr(tokenizer.convert_ids_to_tokens(tid)))

pad_token: <|endoftext|>
eos_token: <|endoftext|>
vocab size: 49152
{'input_ids': [19556, 905, 17], 'attention_mask': [1, 1, 1]}
Total tokens in vocab: 49152

First 5 tokens:
0 -> '<|endoftext|>'
1 -> '<|im_start|>'
2 -> '<|im_end|>'
3 -> '<repo_name>'
4 -> '<reponame>'

Last 5 tokens:
49147 -> 'Lif'
49148 -> 'omia'
49149 -> 'Ġexfol'
49150 -> 'ĠMama'
49151 -> 'ectable'


Write a function `tokenize_helper` that takes an example (using the prompt/response format from the previous step) and produces the following three results:

- `input_ids`: the integer token ids of the concatenated prompt and response;
- `labels`: a list of the same length as `input_ids`, where the response token ids are the same, but where the prompt token ids have all been replaced by the loss masking identifier -100.
- `attention_mask`: the attention mask. This should just be a list of the same length as the other two lists, with all items set to 1.

The reason why `input_ids` and `labels` are different is that
we do not want to compute the training loss for tokens that appear in the user's input. We want to train the model to generate output *conditionally*: based on a prompt. But why the magic number -100? This is the number used by default in PyTorch's [`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) to indicate an item that should be excluded in loss computations. (This issue was also mentioned in [Assignment 1](https://liu-nlp.ai/dl4nlp/units/a1_1.html#task-4.1-implementing-the-trainer).)

In [40]:
def tokenize_helper(example):
    prompt = example['prompt']     # Created in the previous step
    response = example['response'] # Created in the previous step

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(response + tokenizer.eos_token, add_special_tokens=False)["input_ids"]

    input_ids = prompt_ids + response_ids
    labels = [-100] * len(prompt_ids) + response_ids
    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,       # Input token ids of the prompt and response
        "attention_mask": attention_mask,  # Attention mask of the prompt and response
        "labels": labels,          # Output token ids of the prompt (masked) and response
    }

In [41]:
# Inspect one tokenized example to see what tokenize_helper produced.
sample = tokenize_helper(ds_sft["train"][0])

print("Prompt text:")
print(ds_sft["train"][0]["prompt"])
print("Response text:")
print(ds_sft["train"][0]["response"])

print("\nLengths:")
print("  input_ids     :", len(sample["input_ids"]))
print("  labels        :", len(sample["labels"]))
print("  attention_mask:", len(sample["attention_mask"]))

print("\nFirst 15 input_ids:", sample["input_ids"][:15])
print("First 15 labels   :", sample["labels"][:15])
print("First 15 mask     :", sample["attention_mask"][:15])

print("\nLast 10 input_ids:", sample["input_ids"][-10:])
print("Last 10 labels   :", sample["labels"][-10:])

# How many prompt tokens were masked with -100?
num_masked = sum(1 for x in sample["labels"] if x == -100)
print("\nNumber of prompt tokens masked with -100:", num_masked)
print("Number of response tokens (rest):        ", len(sample["labels"]) - num_masked)

# Decode the response part back to text as a sanity check
response_ids = [x for x in sample["labels"] if x != -100]
print("\nDecoded response from labels:")
print(tokenizer.decode(response_ids))

Prompt text:
### Instruction
You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.
Rearrange this sentence to make it easy to understand:
The restaurant ran out of food, so the chef made some more.
### Response

Response text:
The chef made more food after the restaurant ran out.

Lengths:
  input_ids     : 75
  labels        : 75
  attention_mask: 75

First 15 input_ids: [3757, 20880, 198, 2683, 359, 354, 5646, 298, 12021, 11173, 30, 1206, 523, 325, 2711]
First 15 labels   : [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
First 15 mask     : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Last 10 input_ids: [1135, 540, 1114, 990, 260, 15238, 7674, 578, 30, 0]
Last 10 labels   : [1135, 540, 1114, 990, 260, 15238, 7674, 578, 30, 0]

Number of prompt tokens masked with -100: 63
Number of response tokens (rest):         12

Decoded response from labels:
The chef made mo

The whole example (prompt + response + EOS) is 75 tokens long. Three numbers, all equal. That's the first sanity check passed.
Labels are -100 for the 63 prompt tokens (excluded from loss) and match input_ids for the 12 response tokens. Last id is 0 (EOS).

As above, apply the function you implemented to the dataset using `map`. This will add the three new fields to the dataset.

In [42]:
tokenized_ds_sft = ds_sft.map(tokenize_helper)
tokenized_ds_sft

DatasetDict({
    train: Dataset({
        features: ['messages', 'source', 'prompt', 'response', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['messages', 'source', 'prompt', 'response', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 400
    })
})


## Part 2: Evaluation of the baseline model

As a first step, we will see how well the *baseline* model performs: that is, a model that has not been trained to follow instructions.

### ⚙&nbsp; Task 2.1: Preparing for evaluation

In this section, we set up a few utilities we will need to complete our training and evaluation infrastructure. These utilities will be given and you don't need to modify anything.

The first piece we need is a *collator*: that is, a tool that takes a number of instances and creates PyTorch tensors for a training batch. To make the batch fit into rectangular tensors, padding tokens will be added.

In [43]:
def data_collator(batch):
    """
    Create a custom collate function for causal language modeling.

    Args:
        batch: List of examples, each with 'input_ids', 'attention_mask', 'labels'
        tokenizer: Tokenizer with pad_token_id
    """

    input_ids_list = [torch.tensor(example["input_ids"], dtype=torch.long) for example in batch]
    attention_masks_list = [torch.tensor(example["attention_mask"], dtype=torch.long) for example in batch]
    labels_list = [torch.tensor(example['labels'], dtype=torch.long) for example in batch]

    # Find max length in this batch
    max_len = max(x.size(0) for x in input_ids_list)

    # Helper pad function
    def pad_to_max(x_list, pad_value):
        padded = []
        for x in x_list:
            pad_len = max_len - x.size(0)
            if pad_len > 0:
                pad_tensor = torch.full((pad_len,), pad_value, dtype=x.dtype)
                x = torch.cat([x, pad_tensor], dim=0)
            padded.append(x)
        return torch.stack(padded, dim=0)

    # Use tokenizer.pad_token_id for inputs, 0 for attention_mask, -100 for labels
    pad_id = tokenizer.pad_token_id

    batch_input_ids = pad_to_max(input_ids_list, pad_value=pad_id)
    batch_attention_mask = pad_to_max(attention_masks_list, pad_value=0)
    batch_labels = pad_to_max(labels_list, pad_value=-100)

    batch = {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention_mask,
            "labels": batch_labels,
        }
    return batch

The second utility we need is an evaluator. We will use the **ROUGE-L** metric, which computes the longest common subsequence between the model's output and the gold-standard answer. You can read about ROUGE-L here: https://en.wikipedia.org/wiki/ROUGE_(metric)

When using the ROUGE-L metric in a Trainer, we need to wrap it in an object defined as follows:

In [44]:
import evaluate

class RougeMetricComputer:
    """
    Stateful metric for batch_eval_metrics=True.

    It:
      - accumulates predictions and references across batches
      - computes ROUGE-L once at the end (compute_result=True)
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.rouge = evaluate.load("rouge")
        self.all_predictions = []
        self.all_references = []

    def __call__(self, eval_pred, compute_result=False):
        """Accumulate predictions and compute at the end."""

        logits, labels = eval_pred
        pred_ids = logits.argmax(axis=-1)

        # Collect decoded answer-span text from each example in the batch
        for p, lbl in zip(pred_ids, labels):
            mask = lbl != -100
            if mask.sum() == 0:
                continue

            ref_ids = lbl[mask]
            pred_ids_filtered = p[mask]

            ref_text = self.tokenizer.decode(ref_ids, skip_special_tokens=True)
            pred_text = self.tokenizer.decode(pred_ids_filtered, skip_special_tokens=True)

            self.all_references.append(ref_text.strip())
            self.all_predictions.append(pred_text.strip())

        # Only compute at the very end of eval
        if compute_result:
            if len(self.all_references) > 0:
                scores = self.rouge.compute(
                    predictions=self.all_predictions,
                    references=self.all_references,
                )

                # Clear accumulated data for next eval call
                self.all_predictions = []
                self.all_references = []
                return {"rougeL": scores["rougeL"]}
            else:
                return {}
        else:
            return {}

compute_metrics = RougeMetricComputer(tokenizer)

Finally, we make a function that sets up a [`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer).

In [45]:
from transformers import Trainer
from transformers.trainer_callback import ProgressCallback

def make_trainer(model, training_args):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds_sft["train"],
        eval_dataset=tokenized_ds_sft["test"],
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    trainer.callback_handler.callbacks = [
        cb for cb in trainer.callback_handler.callbacks
        if type(cb).__name__ != "NotebookProgressCallback"
    ]
    trainer.add_callback(ProgressCallback)
    return trainer


### 🎓&nbsp; Task 2.2: Evaluating the pre-trained model

Now, we have all the pieces to evaluate our baseline model that has not been instruction-tuned.

The following code will compute the loss on the test set as well as the ROUGE-L score. You will later compare these scores to the models that you train.

Why do you think the ROUGE-L score is as high as it is, even without any training for instruction-following?

In [46]:
from transformers import TrainingArguments
from transformers import AutoModelForCausalLM
import time
import json

print("\n" + "=" * 80)
print("EVALUATING PRETRAINED MODEL")
print("=" * 80)

use_bf16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
use_fp16 = DEVICE == "cuda" and not use_bf16

pretrained_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

pretrained_eval_args = TrainingArguments(
    output_dir="./a3_pretrained_eval",     # Where Trainer logs/artifacts are stored
    eval_strategy="no",                    # No periodic eval during training (we are not training here)
    per_device_eval_batch_size=1,          # Batch size for evaluation
    bf16=use_bf16, fp16=use_fp16,          # Precision mode chosen above
    report_to="none",                      # Disable external logging integrations
    batch_eval_metrics=True,               # Compute metrics with stateful metric object
    eval_accumulation_steps=1,             # Accumulate minimally to reduce memory usage
    do_train=False,                        # Explicitly disable training
    do_eval=True,                          # Enable evaluation
)

pretrained_trainer = make_trainer(pretrained_model, pretrained_eval_args)

# Time the full evaluation pass.
t0 = time.perf_counter()
pretrained_eval_metrics = pretrained_trainer.evaluate()
pretrained_eval_time = time.perf_counter() - t0

# Extract key metrics in convenient scalar form.
pretrained_eval_loss = float(pretrained_eval_metrics["eval_loss"])
pretrained_rougeL = pretrained_eval_metrics.get("eval_rougeL", None)

print("\nPRETRAINED EVAL METRICS:")
print(json.dumps(pretrained_eval_metrics, indent=2))
print(f"Eval time: {pretrained_eval_time:.1f}s")


EVALUATING PRETRAINED MODEL


100%|██████████| 400/400 [00:15<00:00, 26.22it/s]


PRETRAINED EVAL METRICS:
{
  "eval_loss": 1.845808982849121,
  "eval_model_preparation_time": 0.002,
  "eval_rougeL": 0.5996456532784525,
  "eval_runtime": 15.3591,
  "eval_samples_per_second": 26.043,
  "eval_steps_per_second": 26.043,
  "epoch": 0
}
Eval time: 15.4s


In [47]:
import torch

# 1) Pick one test example
idx = 0
example = tokenized_ds_sft["test"][idx]
model_device = next(pretrained_model.parameters()).device

input_ids      = torch.tensor([example["input_ids"]]).to(model_device)
attention_mask = torch.tensor([example["attention_mask"]]).to(model_device)
labels         = torch.tensor([example["labels"]]).to(model_device)

# 2) Teacher-forced forward pass
pretrained_model.eval()
with torch.no_grad():
    outputs = pretrained_model(input_ids=input_ids, attention_mask=attention_mask)
logits = outputs.logits.float()
pred_ids_all = logits.argmax(dim=-1)

# 3) Keep only response positions (labels != -100)
mask     = labels[0] != -100
ref_ids  = labels[0][mask].cpu()
pred_ids = pred_ids_all[0][mask].cpu()

# 4) Decode
prompt_text     = ds_sft["test"][idx]["prompt"]
reference_text  = tokenizer.decode(ref_ids,  skip_special_tokens=True).strip()
prediction_text = tokenizer.decode(pred_ids, skip_special_tokens=True).strip()

# 5) Reuse the same rouge object we already loaded in compute_metrics
score = compute_metrics.rouge.compute(
    predictions=[prediction_text],
    references=[reference_text],
)

print("=" * 80)
print("PROMPT (input given to the model):")
print("=" * 80)
print(prompt_text)

print("\n" + "=" * 80)
print("REFERENCE (ground-truth response):")
print("=" * 80)
print(reference_text)

print("\n" + "=" * 80)
print("PREDICTION (teacher-forced argmax):")
print("=" * 80)
print(prediction_text)

print("\n" + "=" * 80)
print(f"ROUGE-L for this one example: {score['rougeL']:.4f}")
print("=" * 80)

PROMPT (input given to the model):
### Instruction
Correct the verb tense in the following sentence: "I swim every day last week.":
"I swam every day last week."
### Response


REFERENCE (ground-truth response):
"I swam every day last week."

PREDICTION (teacher-forced argmax):
I swim every day last week."
#

ROUGE-L for this one example: 0.8333


## My Reflection:

A high ROUGE-L here does not mean the pretrained model truly follows instructions. In this evaluation, the model gets the gold context and predicts next tokens, so overlap with the reference can be high (around 0.60). ROUGE-L is an overlap metric (0 to 1), so it rewards matching subsequences, not full instruction understanding or semantic quality. When we switch to free generation, the same model often drifts, repeats, or asks new questions instead of answering. So this score reflects token prediction ability more than real instruction-following behavior.

In [48]:
model_device = next(pretrained_model.parameters()).device

questions_to_try = [
    "What is the capital of France?",
    "Explain photosynthesis in one sentence.",
    "Give me three tips to learn Python.",
]

print("=" * 80)
print("PRETRAINED MODEL — FREE GENERATION")
print("=" * 80)

for q in questions_to_try:
    prompt = "### Instruction\n" + q + "\n### Response\n"
    # Tokenize prompt and move tensors
    inputs = tokenizer(prompt, return_tensors="pt").to(model_device)

    pretrained_model.eval()
    with torch.no_grad():
        output_ids = pretrained_model.generate(
            **inputs,
            max_new_tokens=60,                     # Limit response length
            do_sample=False,                       # Greedy (no sampling)
            pad_token_id=tokenizer.pad_token_id,  # Safe padding behavior
            eos_token_id=tokenizer.eos_token_id,  # Stop at EOS if produced
        )

    # Decode full text and keep only the response part after the marker.
    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    generated_part = full_text.split("### Response")[-1].strip()

    print("\n" + "-" * 80)
    print("QUESTION :", q)
    print("MODEL SAID:", generated_part)

PRETRAINED MODEL — FREE GENERATION

--------------------------------------------------------------------------------
QUESTION : What is the capital of France?
MODEL SAID: The capital of France is Paris.
### Instruction
What is the capital of

--------------------------------------------------------------------------------
QUESTION : Explain photosynthesis in one sentence.
MODEL SAID: 

--------------------------------------------------------------------------------
QUESTION : Give me three tips to learn Python.
MODEL SAID: 1. Learn the basics of Python.
2. Learn the basics of programming.
3. Learn the basics of data structures.
4. Learn the basics of algorithms.
5. Learn the basics of data structures.
6. Learn the basics of algorithms.
7. Learn the



## Part 3: Supervised fine-tuning



### 🎓&nbsp; Task 3.1: Training the full model

Next, we train the pre-trained model using SFT over all the parameters, then calculate the metrics and outputs to evaluate how well it follows instructions.

How do the results differ from those in the previous step?

In [24]:
baseline_training_args = TrainingArguments(
    output_dir="./a3_full_sft",            # Directory for Trainer outputs/log files
    eval_strategy="epoch",                 # Run evaluation at the end of each epoch
    logging_steps=2000,                    # Print training logs every N update steps
    save_strategy="no",                    # Do not save checkpoints (keeps disk use low)
    num_train_epochs=1,                    # Train for one full pass over the training set
    per_device_train_batch_size=1,         # Train batch size per device (GPU/CPU)
    per_device_eval_batch_size=1,          # Eval batch size per device
    bf16=use_bf16, fp16=use_fp16,          # Mixed precision mode selected earlier
    report_to="none",                      # Disable external trackers (e.g., wandb)
    batch_eval_metrics=True,               # Use stateful metric accumulation during eval
    eval_accumulation_steps=1,             # Keep eval memory usage minimal
)
# Load a fresh base model for full SFT
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
# Build Trainer using helper
baseline_trainer = make_trainer(base_model, baseline_training_args)
# Train and measure time
t0 = time.perf_counter()
baseline_train_output = baseline_trainer.train()
baseline_train_time = time.perf_counter() - t0
# Evaluate trained model on test split.
baseline_eval_metrics = baseline_trainer.evaluate()
baseline_eval_loss = float(baseline_eval_metrics["eval_loss"])
baseline_rougeL = baseline_eval_metrics.get("eval_rougeL", None)

print("\nFULL SFT TRAINING SUMMARY:")
print(f"Train time: {baseline_train_time:.1f}s")
print(json.dumps({
    "train_loss": float(getattr(baseline_train_output, "training_loss", float("nan"))),
    "eval_loss": baseline_eval_loss,
    "eval_rougeL": baseline_rougeL
}, indent=2))

 40%|████      | 2001/5000 [19:20<29:06,  1.72it/s]

{'loss': '1.428', 'grad_norm': '3.625', 'learning_rate': '3.001e-05', 'epoch': '0.4'}


 80%|████████  | 4001/5000 [38:34<09:56,  1.67it/s]

{'loss': '1.316', 'grad_norm': '8.812', 'learning_rate': '1.001e-05', 'epoch': '0.8'}


100%|██████████| 5000/5000 [48:30<00:00,  1.72it/s]


{'eval_loss': '1.339', 'eval_rougeL': '0.6362', 'eval_runtime': '16.12', 'eval_samples_per_second': '24.82', 'eval_steps_per_second': '24.82', 'epoch': '1'}
{'train_runtime': '2910', 'train_samples_per_second': '1.718', 'train_steps_per_second': '1.718', 'train_loss': '1.355', 'epoch': '1'}


100%|██████████| 400/400 [00:22<00:00, 17.89it/s]


FULL SFT TRAINING SUMMARY:
Train time: 2910.2s
{
  "train_loss": 1.3550121826171875,
  "eval_loss": 1.3386359214782715,
  "eval_rougeL": 0.6361691127770766
}


## My reflection:

ROUGE-L improved only slightly from 0.60 to 0.64. The small ROUGE-L gain is expected because teacher forcing on masked-prompt evaluation already gives a strong pretrained baseline.

### ⚙&nbsp; Task 3.3: Counting the number of trainable parameters

Define a function `num_trainable_parameters` that computes the number of floating-point numbers that a given model will update during training.

**Hints**:
- For a PyTorch module `m`, you can use `m.parameters()` to access its parameter tensors.
- However, you should only include parameter tensors where the flag `requires_grad` is True.


In [25]:
def num_trainable_parameters(model):
    """Count number of trainable parameters.

    Args:
        model: A PyTorch module.
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

Apply this function to the SFT-trained model and check that the result makes sense.

In [26]:
print(f"Full SFT trainable parameters: {num_trainable_parameters(base_model):,}")

Full SFT trainable parameters: 134,515,008


## My Reflection:
On this model this returns ~134.5 M (the whole SmolLM2 model) because nothing was frozen. In Part 4 the same function on the LoRA model will return only a few hundred thousand, which quantifies LoRA's efficiency.

## Part 4: Parameter-efficient fine-tuning

In the last section of this assignment, we will use LoRA to train the model in a more parameter-efficient manner. You may want to prepare by reading  by [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685) and the teaching material provided for this course.

### ⚙&nbsp; Task 4.1: Utilities for modifying models

Define a function `extract_lora_targets` that extracts the relevant linear layers from all Transformer blocks in your selected LLM.
It is up to you to decide what layers to select; in the experiments described in the original LoRA paper, the query and value projection matrices were fine-tuned with LoRA, while all other layers were left unchanged.
Return a dictionary that maps the component name to the corresponding linear layer.

As we saw earlier (in Assignment 2 and elsewhere), a Transformer model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. You can use get_submodule() to retrieve a layer by a string name. This name depends on the model you have selected. For instance, in the `SmolLM2-135M` model, `'model.layers.0.self_attn.q_proj'`
 refers to the query projection in Transformer layer 0.

It is OK to hard-code this part, so that you just enumerate the layers you want to extract. Alternatively, use a utility such as `model.named_modules()` to iterate through the model's layers.

In [27]:
def extract_lora_targets(model):
    targets = {}
    # Iterate over all named submodules in the model hierarchy.
    for name, module in model.named_modules():
        # Keep only linear layers used in attention projections:
        # q_proj (query), k_proj (key), v_proj (value), o_proj (output).
        if isinstance(module, torch.nn.Linear) and any(
            name.endswith(suffix) for suffix in ("q_proj", "k_proj", "v_proj", "o_proj")
        ):
            # Store the layer by its full name so it can be replaced later.
            targets[name] = module
    # Return all LoRA candidate layers.
    return targets

We also need a convenience function that puts layers back into a model. The following function does the trick. The `named_layers` argument uses the same format as returned by `extract_lora_targets`.

In [28]:
def replace_layers(model, named_layers):
    """
    Replace submodules in `model` by name.
    """
    for name, layer in named_layers.items():
        components = name.split(".")
        submodule = model
        for comp in components[:-1]:
            submodule = getattr(submodule, comp)
        # Replace the target child module with the provided layer.
        setattr(submodule, components[-1], layer)
    # Return the modified model (in-place replacement already done).
    return model

### 🎓&nbsp; Task 4.2: Implementing the LoRA layer

To implement the LoRA approach, we define a new type of layer that will be used as a drop-in replacement for a regular linear layer.

In [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685), the structure is presented visually in Figure 1, and equation (3) shows the same idea.

Start from the following skeleton and fill in the missing pieces:


In [ ]:
import torch.nn as nn

class LoRALayer(nn.Module):
    def __init__(self, W, r, alpha):
        super().__init__()
        self.W = W
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # Freeze original weight/bias and train only low-rank adapters.
        for p in self.W.parameters():
            p.requires_grad = False
        # Dimensions from original linear layer
        in_features = W.in_features
        out_features = W.out_features
        # LoRA parameters:
        self.A = nn.Parameter(torch.zeros(r, in_features))
        self.B = nn.Parameter(torch.zeros(out_features, r))

        nn.init.kaiming_uniform_(self.A, a=5**0.5)
        nn.init.zeros_(self.B)

    def forward(self, x):
        # Base output from frozen original layer
        base = self.W(x)
        # Match dtype with input
        A = self.A.to(x.dtype)
        B = self.B.to(x.dtype)
        # Low-rank update
        lora_update = (x @ A.t()) @ B.t()
        return base + self.scaling * lora_update

Here, `W` is the linear layer we are fine-tuning, while `r` and `alpha` are hyperparameters described in section 4.1. of the paper. The `r` parameter controls the parameter efficiency: by setting it to a low value, we save memory but make a rougher approximation. The `alpha` parameter is a scaling factor.

### 🎓&nbsp; Task 4.3: Fine-tuning with LoRA

Set up a model where you replace the four linear layers in attention blocks (query, key, value, and output) with LoRA layers. Use the following steps:
- First use `extract_lora_targets` to get the relevant linear layers.
- Each of the linear layers in the returned dictionary should be wrapped inside a LoRA layer.
- Then use `replace_layers` to put them back into the model.

Train this model and compare the training speed, metrics, and outputs to the results from Part 3.

Apply your parameter counting function (`num_trainable_parameters`) to this model, compare the results to those in Part 3, and make sure that these results correspond to your expectations.


### 🎓&nbsp; Task 4.4: Qualitative inspection

Run the three models interactively on some examples of your own choice (either taken from the training or test sets, or created by yourself). The convenience function below can be of use, but you need to complete it by using the prompt format you defined in Task 1.2.

Do your models seem to have learned the instruction-following behavior (at least to some extent)? Do they respond to user queries sensibly?

The quality we see here will depend on your choice of base model as well as how much you trained it.

In [30]:
# Parameter counts
full_sft_trainable = num_trainable_parameters(base_model)
print(f"Full SFT trainable params: {full_sft_trainable:,}")

# Build LoRA model
lora_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
targets = extract_lora_targets(lora_model)
print(f"LoRA target linear layers: {len(targets)}")

r = 8
alpha = 16
lora_wrapped = {name: LoRALayer(layer, r=r, alpha=alpha) for name, layer in targets.items()}
lora_model = replace_layers(lora_model, lora_wrapped).to(DEVICE)

# Ensure only LoRA adapters are trainable
for p in lora_model.parameters():
    p.requires_grad = False
for module in lora_model.modules():
    if isinstance(module, LoRALayer):
        module.A.requires_grad = True
        module.B.requires_grad = True

lora_trainable = num_trainable_parameters(lora_model)
print(f"LoRA trainable params: {lora_trainable:,} ({100*lora_trainable/full_sft_trainable:.2f}% of full SFT)")

lora_training_args = TrainingArguments(
    output_dir="./a3_lora_sft",       # Folder for Trainer outputs
    eval_strategy="epoch",            # Evaluate once per epoch
    logging_steps=2000,               # Log every N optimizer steps
    save_strategy="no",               # No checkpoints (save disk/time)
    num_train_epochs=1,               # One epoch for this assignment setup
    per_device_train_batch_size=1,    # Batch size per device during training
    per_device_eval_batch_size=1,     # Batch size per device during evaluation
    bf16=use_bf16, fp16=use_fp16,     # Mixed precision (auto-selected earlier)
    report_to="none",                 # Disable external logging tools
    batch_eval_metrics=True,          # Needed for stateful metric wrapper
    eval_accumulation_steps=1,        # Keep eval memory footprint low
)

# Build Trainer and run LoRA training + evaluation
lora_trainer = make_trainer(lora_model, lora_training_args)

t0 = time.perf_counter()
_ = lora_trainer.train()
lora_train_time = time.perf_counter() - t0
lora_eval_metrics = lora_trainer.evaluate()
lora_eval_loss = float(lora_eval_metrics["eval_loss"])
lora_rougeL = lora_eval_metrics.get("eval_rougeL", None)

print("\nLORA SFT SUMMARY:")
print(f"Train time: {lora_train_time:.1f}s")
print(json.dumps({
    "eval_loss": lora_eval_loss,
    "eval_rougeL": lora_rougeL,
    "trainable_params": lora_trainable
}, indent=2))

def build_prompt(user_text, system_text="You are a helpful assistant."):
    return f"### Instruction\n{system_text}\n{user_text}\n### Response\n"

def generate_response(model, user_text, max_new_tokens=96):
    model.eval()
    prompt = build_prompt(user_text)
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(generated[0], skip_special_tokens=True)
    # Return only the portion after the response marker
    return text.split("### Response")[-1].strip()

test_prompts = [
    "Explain in simple terms why overfitting is a problem in machine learning.",
    "Write a short polite email asking for a deadline extension.",
    "Give three tips for learning Python effectively."
]

print("\nQUALITATIVE COMPARISON")
for q in test_prompts:
    print("\n" + "-" * 80)
    print("USER:", q)
    print("PRETRAINED:", generate_response(pretrained_model, q))
    print("FULL SFT:", generate_response(base_model, q))
    print("LORA SFT:", generate_response(lora_model, q))

Full SFT trainable params: 134,515,008


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 5544.03it/s]


LoRA target linear layers: 120
LoRA trainable params: 921,600 (0.69% of full SFT)


 40%|████      | 2001/5000 [04:39<06:33,  7.63it/s]

{'loss': '1.466', 'grad_norm': '0.8958', 'learning_rate': '3.001e-05', 'epoch': '0.4'}


...

{'loss': '1.322', 'grad_norm': '1.757', 'learning_rate': '1.001e-05', 'epoch': '0.8'}


100%|██████████| 5000/5000 [11:40<00:00,  7.13it/s]


{'eval_loss': '1.336', 'eval_rougeL': '0.6351', 'eval_runtime': '31.73', 'eval_samples_per_second': '12.61', 'eval_steps_per_second': '12.61', 'epoch': '1'}
{'train_runtime': '701', 'train_samples_per_second': '7.133', 'train_steps_per_second': '7.133', 'train_loss': '1.371', 'epoch': '1'}


100%|██████████| 400/400 [00:18<00:00, 21.52it/s]



LORA SFT SUMMARY:
Train time: 701.2s
{
  "eval_loss": 1.3363733291625977,
  "eval_rougeL": 0.6351097655958811,
  "trainable_params": 921600
}

QUALITATIVE COMPARISON

--------------------------------------------------------------------------------
USER: Explain in simple terms why overfitting is a problem in machine learning.
PRETRAINED: You are a helpful assistant.
Explain in simple terms why overfitting is a problem in machine learning.
### Feedback
You are a helpful assistant.
Explain in simple terms why overfitting is a problem in machine learning.
### Feedback
You are a helpful assistant.
Explain in simple terms why overfitting is a problem in machine learning.
### Feedback
You are a helpful assistant.
Explain in simple terms why overfitting is a problem in machine learning.
FULL SFT: Overfitting occurs when a model learns the training data too well, and it starts to overfit the training data. Overfitting can lead to poor performance in the training data, and it can also lead to 

## My Reflection on 4.3:
LoRA trained only 921 k parameters (about 0.68 % of full SFT's 134.5 M) finished in ~11 min vs ~48 min (4× faster), yet reached essentially the same eval_loss (1.336 vs 1.338) and ROUGE-L (0.636 vs 0.637): same quality, ~1 % of the cost.

## My Reflection on 4.4:

Yes, the qualitative outputs suggest that instruction-following was learned to some extent by both fine-tuned models.  
The pretrained model mostly failed (model repeats part of the input prompt, model output is cut off too early, and model stops following the required template), so it did not respond sensibly.  
In contrast, Full SFT and LoRA SFT produced relevant answers for all three prompts (explanation, email, and list of tips), which indicates improved instruction following.

Between the two fine-tuned models, Full SFT and LoRA were similar in quality, with LoRA slightly more concise and sometimes repetitive. Overall, both were useful but still generic and occasionally awkward, which is expected from a small base model and only 1 epoch of training.